# 02. Xe II: сечения и константы скоростей линий, предварительная CRM

По сечениям Ванга (возбуждение верхних уровней) и ветвлениям из ноутбука 01:

* **σ линии** = BR · σ(возбуждения) из основного 5p⁵ ²P₃/₂, из ²P₁/₂ и из метастабилей 6s, 5d;
* **константы скорости линий** k_line(Te) для максвелловской ФРЭЭ и ФРЭЭ Бугровой;
* **предварительная CRM** (уровни 1–56 + (¹S₀)6p) с временем пролёта τ: интенсивности
  линий и их чувствительность к ne и Te → выбор линий-диагностик.

Каскадов с 6d/7s/4f и переходов метастабиль↔метастабиль у Ванга нет — их добавит
ноутбук 03 (тогда CRM здесь можно пересобрать с `crm_xe2_ext.json`).

In [ ]:
# ---------- параметры (поменяйте пути под себя)
from pathlib import Path
import sys
REPO    = Path("/data/pydbsr_calc/DBSR_src")          # клон репозитория pydbsr
CRM     = REPO / "examples" / "crm"                   # эта папка: nbtools.py, run_model.py, models.py
DATA    = Path("/data/pydbsr_calc/data")              # ваши файлы
XLSX    = DATA / "CrossSectionsIon_3.xlsx"            # Wang et al (2019), сечения e + Xe+
SPECTRUM, SHEET = DATA / "A25_second.xlsx", "data2"   # измеренный спектр
NIST    = REPO / "docs" / "crm"                       # выгрузки NIST (nist_*.csv) и список линий
RUNS    = Path("/data/pydbsr_calc/runs/crm")          # результаты расчётов (большой диск)
SCRATCH, SCRATCH_GB = Path.home() / "pydbsr_tmp", 5   # временные файлы dbsr_mat (быстрый диск)
CORES, MEM, HD_THREADS = 32, 100, 8                   # ядра, ГБ памяти, потоки диагонализации
sys.path.insert(0, str(CRM))
import numpy as np, matplotlib.pyplot as plt
import pydbsr as db
from pydbsr import crm, reference
import nbtools as nb
%matplotlib inline

In [ ]:
ref = reference.read_xlsx(XLSX)
W = RUNS / "xe2_wang"
br, tot, A = nb.branching_from_csv(W / "transitions_E1.csv")        # из ноутбука 01
lines = [r for r in nb.read_csv(NIST / "lines.csv") if r["ion"] == "Xe II" and r["class"] in ("A", "B")]
print(len(lines), "линий класса A/B")

## Сечения линий: запрошенные 452.4, 659.5, 669.4 и любые другие

In [ ]:
SHOW = [452.421, 659.501, 669.432, 561.667, 545.045, 545.090]
fig, axs = plt.subplots(2, 3, figsize=(14, 7))
for ax, wl in zip(axs.flat, SHOW):
    r = min(lines, key=lambda r: abs(float(r["wl_nm"]) - wl))
    u, l = int(r["upper_no"]), int(r["lower_no"])
    b = br[u][l]
    for i, lab in ((1, "из ²P₃/₂"), (2, "из ²P₁/₂")):
        if (i, u) in ref.sigma:
            E, s = nb.line_sigma(ref, u, b, i)
            ax.plot(E, s / 1e-16, lw=0.8, label=lab)
    ax.set_xscale("log"); ax.set_title(f"{r['wl_nm']} нм  ({u}→{l}), BR = {b:.3f}", fontsize=9)
    ax.set_xlabel("E, эВ"); ax.set_ylabel("σ линии, 10⁻¹⁶ см²")
axs.flat[0].legend(); fig.tight_layout(); plt.show()

## Таблица: сечения всех линий класса A/B → `xe2_line_sigma.xlsx`

Лист на линию: энергия, σ линии из ²P₃/₂, из ²P₁/₂ и из каждого метастабиля (в 10⁻¹⁶ см²), как
у Ванга. Плюс лист `rates` с k_line(Te).

In [ ]:
import openpyxl
TE = [0.5, 0.7, 1.0, 1.5, 2.0, 3.0, 5.0]
wb = openpyxl.Workbook(); wr = wb.active; wr.title = "rates"
wr.append(["λ, нм", "верхний", "нижний", "BR"] + [f"k(²P3/2), Te={t} эВ" for t in TE] + ["Σk(метаст.)/k(осн.), 1.1 эВ"])
for r in lines:
    u, l = int(r["upper_no"]), (int(r["lower_no"]) if r["lower_no"] else None)
    b = br.get(u, {}).get(l)
    if not b:
        continue
    ws = wb.create_sheet(f"{float(r['wl_nm']):.3f}")
    init = sorted(i for (i, j) in ref.sigma if j == u)
    for i in init:
        E, s = ref.sigma[(i, u)]
        ws.append([f"{i}->{u}", ref.label(i)]); ws.append(["Energy(eV)", "Sigma_line(1E-16 cm^2)"])
        for e, x in zip(E, s * b / 1e-16):
            ws.append([float(e), float(x)])
        ws.append([])
    E, s = ref.sigma[(1, u)]
    thr = ref.level(u).energy_cm / nb.CM_PER_EV
    k = [b * crm.rate_from_sigma(E, s, t, thr) for t in TE]
    km = sum(crm.rate_from_sigma(*ref.sigma[(i, u)], 1.1, (ref.level(u).energy_cm - ref.level(i).energy_cm) / nb.CM_PER_EV)
             for i in init if i > 2)
    wr.append([float(r["wl_nm"]), r["upper"], r["lower"], b] + k + [km / crm.rate_from_sigma(E, s, 1.1, thr)])
wb.save(W / "xe2_line_sigma.xlsx"); print("->", W / "xe2_line_sigma.xlsx")

## Предварительная CRM на данных Ванга

Уровни 1–56, 101, 105. Возбуждение: всё, что есть у Ванга (основное, ²P₁/₂ → всё;
метастабили → 6p); девозбуждение — детальный баланс. Радиация — A из ноутбука 01
(E1 + M1/E2 для метастабилей). Сток всех уровней — пролёт τ (ион пролетает зону
наблюдения за ~L/v).

**Ограничения**: нет переходов метастабиль↔метастабиль, каскадов с 6d/7s/4f и
ионизации. Числа предварительные; цель — увидеть, какие линии чувствительны к ne.

In [ ]:
TE_GRID = np.array([0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0, 5.0, 7.0, 10.0])
keep = list(range(1, 57)) + [101, 105]
lv = {i: ref.level(i) for i in keep}
data = crm.CRMData("Xe II", [str(i) for i in keep], [lv[i].two_j + 1 for i in keep],
                   [lv[i].energy_cm for i in keep], [ref.label(i) for i in keep], TE_GRID)
for (i, j), (E, s) in ref.sigma.items():
    if i in lv and j in lv and lv[i].energy_cm < lv[j].energy_cm:
        thr = (lv[j].energy_cm - lv[i].energy_cm) / nb.CM_PER_EV
        data.add_rate(str(i), str(j), [crm.rate_from_sigma(E, s, t, thr) for t in TE_GRID], "Wang2019")
for kind in ("E1", "M1", "E2"):
    f = W / f"transitions_{kind}.csv"
    if f.exists():
        for u, d in nb.branching_from_csv(f)[2].items():
            for l, a in d.items():
                if u in lv and l in lv:
                    data.add_A(str(u), str(l), a + data.A.get((data.index[str(u)], data.index[str(l)]), 0.0))
data.save(W / "crm_xe2_wang_prelim.json")
print(len(data.k), "констант скорости,", len(data.A), "A")

In [ ]:
TAU = 3e-6          # с: время пролёта иона Xe+ через зону наблюдения (L ~ 1 см, v ~ 3 км/с)
SEL = [(r["upper_no"], r["lower_no"], r["wl_nm"]) for r in lines if r.get("role") and r["lower_no"]]
pairs = [(u, l) for u, l, _ in SEL]
print(f"{'λ, нм':>8} {'верх':>4}   " + "   ".join(f"ne={ne:.0e}: dlnI/dlnne dlnI/dlnTe" for ne in (1e10, 1e11, 1e12)))
res = {ne: crm.sensitivity(data, pairs, ne, 1.1, tau=TAU) for ne in (1e10, 1e11, 1e12)}
for u, l, wl in SEL:
    print(f"{wl:>8} {u:>4}   " + "   ".join(f"{res[ne][(u, l)]['dlnI_dlnne']:24.2f} {res[ne][(u, l)]['dlnI_dlnTe']:11.2f}"
                                         for ne in res))

## Отношения линий против ne и Te (карта для диагностики)

In [ ]:
RATIOS = [(("41", "15"), ("44", "16"), "561.7/545.0"), (("31", "7"), ("29", "7"), "547.3/553.1"),
          (("52", "33"), ("56", "28"), "659.5/504.5"), (("49", "28"), ("30", "9"), "627.1/594.6")]
NE = np.logspace(9.5, 12.5, 25); TES = [0.8, 1.1, 1.5, 2.0, 3.0]
fig, axs = plt.subplots(1, len(RATIOS), figsize=(16, 3.8))
for ax, (a, b, lab) in zip(axs, RATIOS):
    for te in TES:
        y = []
        for ne in NE:
            p = crm.steady_state(data, ne, te, tau=TAU)
            y.append(crm.line_emission(data, p, *a) / crm.line_emission(data, p, *b))
        ax.loglog(NE, y, label=f"Te = {te} эВ")
    ax.set_title(lab); ax.set_xlabel("ne, см⁻³")
axs[0].set_ylabel("отношение (фотоны)"); axs[0].legend(fontsize=7); fig.tight_layout(); plt.show()